# Query Structuring

**Query Structuring** is the process of converting a user's natural-language query into a structured format that a retrieval system can understand and use for more precise searching.

### Example

```text
User Query:
"Find Python tutorials from 2025"

        ↓

Structured Query:
{
    "topic": "Python",
    "type": "tutorial",
    "year": 2025
}
```

### In Simple Words

> Query Structuring converts a natural-language question into **structured search instructions** to improve retrieval accuracy.


In [1]:
from yt_dlp import YoutubeDL
from langchain_community.document_loaders import YoutubeLoader


def patched_get_video_info(self):
    """Get YouTube metadata using yt-dlp."""

    url = f"https://www.youtube.com/watch?v={self.video_id}"

    ydl_opts = {
        "quiet": True,
        "no_warnings": True,
        "skip_download": True,
    }

    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)

    return {
        "title": info.get("title"),
        "description": info.get("description"),
        "view_count": info.get("view_count"),
        "publish_date": info.get("upload_date"),
        "length": info.get("duration"),
        "author": info.get("uploader"),
        "thumbnail_url": info.get("thumbnail"),
    }


# Patch LangChain's metadata method
YoutubeLoader._get_video_info = patched_get_video_info

C:\Users\Acer\AppData\Local\Temp\ipykernel_11340\98442337.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import YoutubeLoader


In [2]:
from langchain_community.document_loaders import YoutubeLoader

docs = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=pbAd8O1Lvm4", add_video_info=True
).load()

docs[0].metadata

{'source': 'pbAd8O1Lvm4',
 'title': 'Self-reflective RAG with LangGraph: Self-RAG and CRAG',
 'description': 'Self-reflection can greatly enhance RAG, enabling correction of poor quality retrieval or generations. Several recent RAG papers focus on this theme, but implementing the ideas can be tricky. Here, we show that LangGraph can be easily used for "flow engineering" of self-reflective RAG pipelines. We provide cookbooks for implementing ideas from two interesting papers, Self-RAG and C-RAG.\n\nCode:\nhttps://github.com/langchain-ai/langgraph/tree/main/examples/rag',
 'view_count': 38911,
 'publish_date': '20240207',
 'length': 1058,
 'author': 'LangChain',
 'thumbnail_url': 'https://i.ytimg.com/vi/pbAd8O1Lvm4/maxresdefault.jpg'}

In [3]:
import datetime
from typing import Optional

from pydantic import BaseModel, Field


class TutorialSearch(BaseModel):
    """Tutorial videos ko database ma structured search garne model."""

    # Video transcript ma similarity search garne query
    content_search: str = Field(
        ...,
        description="Similarity search query applied to video transcripts.",
    )

    # Video title ma search garne short query
    title_search: str = Field(
        ...,
        description=(
            "Short search query containing keywords that may appear "
            "in the video title."
        ),
    )

    # Minimum views filter
    min_view_count: Optional[int] = Field(
        None,
        description=("Minimum view count filter. " "Only use if explicitly specified."),
    )

    # Maximum views filter
    max_view_count: Optional[int] = Field(
        None,
        description=("Maximum view count filter. " "Only use if explicitly specified."),
    )

    # Minimum publication date
    earliest_publish_date: Optional[datetime.date] = Field(
        None,
        description=(
            "Earliest publish date filter. " "Only use if explicitly specified."
        ),
    )

    # Maximum publication date
    latest_publish_date: Optional[datetime.date] = Field(
        None,
        description=(
            "Latest publish date filter. " "Only use if explicitly specified."
        ),
    )

    # Minimum video length
    min_length_sec: Optional[int] = Field(
        None,
        description=(
            "Minimum video length in seconds. " "Only use if explicitly specified."
        ),
    )

    # Maximum video length
    max_length_sec: Optional[int] = Field(
        None,
        description=(
            "Maximum video length in seconds. " "Only use if explicitly specified."
        ),
    )

    # Search result lai readable format ma print garne
    def pretty_print(self) -> None:
        for field_name, value in self.model_dump().items():
            if value is not None:
                print(f"{field_name}: {value}")

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

# User ko natural-language question lai database query ma
# convert garna system prompt create gareko
system = """
You are an expert at converting user questions into structured database queries.

You have access to a database of tutorial videos about a software library
for building LLM-powered applications.

Given a user question, return a structured query optimized to retrieve
the most relevant results.

If there are acronyms or words you are not familiar with,
do not try to rephrase them.
"""


# System prompt ra user question combine gareko
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)


# Local Ollama LLM use gareko
llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)


# TutorialSearch schema anusar structured output generate gareko
structured_llm = llm.with_structured_output(TutorialSearch)


# Query analyzer chain create gareko
query_analyzer = prompt | structured_llm

In [5]:
# Natural-language query lai structured query ma convert gareko
result = query_analyzer.invoke({"question": "rag from scratch"})

# Structured query lai readable format ma print gareko
result.pretty_print()

content_search: SELECT * FROM tutorial_videos WHERE title LIKE '%build LLM-powered application%' OR description LIKE '%build LLM-powered application%' OR tags LIKE '%build LLM-powered application%'
title_search: SELECT * FROM tutorial_videos WHERE title LIKE '%from scratch%'
min_length_sec: 0
max_length_sec: 3600


In [6]:
# Natural-language query lai structured query ma convert gareko
result = query_analyzer.invoke(
    {"question": "videos on chat langchain published in 2023"}
)

# Generated structured query lai readable format ma print gareko
result.pretty_print()

content_search: SELECT * FROM videos WHERE (title LIKE '%chat langchain%' OR description LIKE '%chat langchain%') AND (published_date >= '2023-01-01' AND published_date <= '2023-12-31') ORDER BY relevance DESC;
title_search: Chat Langchain
min_length_sec: 60
max_length_sec: 3600


In [7]:
# Topic ra publication date filter bhayeko query test gareko
result = query_analyzer.invoke(
    {
        "question": (
            "videos that are focused on the topic of chat langchain "
            "that are published before 2024"
        )
    }
)

# Structured query readable format ma print gareko
result.pretty_print()

content_search: go ahead and query the database for videos that match these criteria: title contains 'chat' AND title contains 'langchain' AND publication_date <= '2023-12-31' AND NOT (title contains 'tutorial' OR title contains 'getting started') ORDER BY relevance DESC LIMIT 10
title_search: Chat Langchain
max_length_sec: 3600


In [8]:
# Topic ra video length filter bhayeko query test gareko
result = query_analyzer.invoke(
    {
        "question": (
            "how to use multi-modal models in an agent, " "only videos under 5 minutes"
        )
    }
)

# Structured query readable format ma print gareko
result.pretty_print()

content_search: SELECT * FROM tutorial_videos WHERE topic='multi-modal models' AND duration<300 AND length<500
title_search: SELECT title FROM tutorial_videos WHERE topic='multi-modal models' AND duration<300 AND length<500


In [9]:
import inspect
from langchain_community.document_loaders import YoutubeLoader

print(inspect.signature(YoutubeLoader._get_video_info))
print(inspect.getsource(YoutubeLoader._get_video_info))

(self)
def patched_get_video_info(self):
    """Get YouTube metadata using yt-dlp."""

    url = f"https://www.youtube.com/watch?v={self.video_id}"

    ydl_opts = {
        "quiet": True,
        "no_warnings": True,
        "skip_download": True,
    }

    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)

    return {
        "title": info.get("title"),
        "description": info.get("description"),
        "view_count": info.get("view_count"),
        "publish_date": info.get("upload_date"),
        "length": info.get("duration"),
        "author": info.get("uploader"),
        "thumbnail_url": info.get("thumbnail"),
    }

